# Database setup

Reads a v2 JSON experiment config, loads it into a DuckDB database through the `deepecohab.utils.db` / `deepecohab.utils.loaders` domain layer, then extracts the experiment, cages, animals, and antenna-pair interpolation data back out.

In [1]:
from pathlib import Path

import polars as pl

from deepecohab.utils.domain import Cage
from deepecohab.utils.loaders import JsonConfigLoader
from deepecohab.utils.db import connect, DuckDBExperimentRepository

## Read the config

In [2]:
config_path = Path.home() / "Downloads" / "DeepEcoHAB_config.json"
loader = JsonConfigLoader(config_path)

experiment = loader.load_experiment(interpolate=True)
experiment

Experiment(name='DeepEcoHAB', start=datetime.datetime(2026, 6, 10, 11, 39, 24, 553118), end=None, light_start='07:00', dark_start='19:00', recording_timezone='Europe/Warsaw', animals={'EE1CEC1A': Animal(tag_no='EE1CEC1A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), '6D68A819': Animal(tag_no='6D68A819', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), 'DD43E61A': Animal(tag_no='DD43E61A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), '3AFE9F1A': Animal(tag_no='3AFE9F1A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', tre

## Store in the database

In [3]:
db_path = Path.cwd() / "ecohab.duckdb"
con = connect(db_path)
repo = DuckDBExperimentRepository(con)

if not repo.exists(experiment.name):
    repo.save(experiment, loader.layout_cfg)

repo.exists(experiment.name)

True

## Extract the experiment

In [4]:
loaded = repo.get(experiment.name, interpolate=True)
loaded

Experiment(name='DeepEcoHAB', start=datetime.datetime(2026, 6, 10, 11, 39, 24, 553118, tzinfo=<DstTzInfo 'Europe/Warsaw' CEST+2:00:00 DST>), end=None, light_start='07:00', dark_start='19:00', recording_timezone='Europe/Warsaw', animals={'EE1CEC1A': Animal(tag_no='EE1CEC1A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), '6D68A819': Animal(tag_no='6D68A819', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), 'DD43E61A': Animal(tag_no='DD43E61A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic_background=None, mouse_line=None, genotype='WT', treatment='Neurolux+', notes='Cohort1_implant_worked_03_06_2026'), '3AFE9F1A': Animal(tag_no='3AFE9F1A', sex='Male', age={'age': '6', 'unit': 'months'}, dob=None, genetic

## Extract animals

In [5]:
pl.DataFrame([
    {
        "tag_no": a.tag_no, "sex": a.sex, "genotype": a.genotype, "treatment": a.treatment,
        "genetic_background": a.genetic_background, "mouse_line": a.mouse_line,
        "age_value": (a.age or {}).get("age"), "age_unit": (a.age or {}).get("unit"),
        "dob": a.dob, "notes": a.notes,
    }
    for a in loaded.animals.values()
])

tag_no,sex,genotype,treatment,genetic_background,mouse_line,age_value,age_unit,dob,notes
str,str,str,str,null,null,str,str,null,str
"""EE1CEC1A""","""Male""","""WT""","""Neurolux+""",null,null,"""6""","""months""",null,"""Cohort1_implant_worked_03_06_2…"
"""6D68A819""","""Male""","""WT""","""Neurolux+""",null,null,"""6""","""months""",null,"""Cohort1_implant_worked_03_06_2…"
"""DD43E61A""","""Male""","""WT""","""Neurolux+""",null,null,"""6""","""months""",null,"""Cohort1_implant_worked_03_06_2…"
"""3AFE9F1A""","""Male""","""WT""","""Neurolux+""",null,null,"""6""","""months""",null,"""Cohort1_implant_worked_03_06_2…"
"""D5969C1A""","""Male""","""WT""","""Neurolux+""",null,null,"""6""","""months""",null,"""Cohort1_implant_worked_03_06_2…"
…,…,…,…,…,…,…,…,…,…
"""D71EA819""","""Male""","""WT""","""No_implant""",null,null,"""6""","""months""",null,"""No_implant_WT_C57_from_Ksenia"""
"""E967A819""","""Male""","""WT""","""No_implant""",null,null,"""6""","""months""",null,"""No_implant_WT_C57_from_Ksenia"""
"""E6A59C1A""","""Male""","""WT""","""No_implant""",null,null,"""6""","""months""",null,"""No_implant_WT_C57_from_Ksenia"""


## Extract cages

In [6]:
loaded.layout.to_dimension_frames()["cages"]

cage_id,cage_no,cage_type
str,i64,str
"""cage_1""",1,"""Stimulus"""
"""cage_2""",2,"""Food"""
"""cage_3""",3,"""Stimulus"""
"""cage_4""",4,"""Food"""


## Extract interpolation data

The layout resolves every observed antenna pair to either a `Cage` (animal stayed in place) or a `Crossing` (animal moved between cages through a tunnel). This pair → location map is what `interpolate=True` widens: reads that skip an antenna are filled in using it.

In [7]:
def describe_pair(target):
    if isinstance(target, Cage):
        return "cage", target.id, None
    return "crossing", None, target.tunnel_id

rows = []
for (a_from, a_to), target in sorted(loaded.layout.pairs.items()):
    kind, cage_id, tunnel_id = describe_pair(target)
    rows.append({
        "antenna_from": a_from, "antenna_to": a_to, "kind": kind,
        "cage_id": cage_id, "tunnel_id": tunnel_id,
    })

pl.DataFrame(rows)

antenna_from,antenna_to,kind,cage_id,tunnel_id
i64,i64,str,str,str
0,0,"""cage""","""cage_1""",null
0,1,"""crossing""",null,"""tunnel_1"""
0,7,"""cage""","""cage_2""",null
0,8,"""cage""","""cage_1""",null
0,9,"""cage""","""cage_1""",null
…,…,…,…,…
9,0,"""cage""","""cage_1""",null
9,4,"""cage""","""cage_4""",null
9,5,"""cage""","""cage_4""",null


### Interpolated vs. raw antenna pairs

In [8]:
raw_layout = loader.load_layout(interpolate=False)
print(f"raw antenna pairs: {len(raw_layout.pairs)}")
print(f"interpolated antenna pairs: {len(loaded.layout.pairs)}")

raw antenna pairs: 24
interpolated antenna pairs: 40
